In [1]:
import numpy as np

from Util.Problems import Problem, solution

class P008(Problem):
    number = 8
    title = "Largest Product in a Series"
    description = """<p>The four adjacent digits in the $1000$-digit number that have the greatest product are $9 \\times 9 \\times 8 \\times 9 = 5832$.</p><p class="monospace break_word copy_to_clipboard">
731671765313306249192251196744265747423553491949349698352031277450632623957831801698480186947885184385861560789112949495459501737958331952853208805511125406987471585238630507156932909632952274430435576689664895044524452316173185640309871112172238311362229893423380308135336276614282806444486645238749303589072962904915604407723907138105158593079608667017242712188399879790879227492190169972088809377665727333001053367881220235421809751254540594752243525849077116705560136048395864467063244157221553975369781797784617406495514929086256932197846862248283972241375657056057490261407972968652414535100474821663704844031<span class="red">9989</span>000889524345065854122758866688116427171479924442928230863465674813919123162824586178664583591245665294765456828489128831426076900422421902267105562632111110937054421750694165896040807198403850962455444362981230987879927244284909188845801561660979191338754992005240636899125607176060588611646710940507754100225698315520005593572972571636269561882670428252483600823257530420752963450</p><p>Find the thirteen adjacent digits in the $1000$-digit number that have the greatest product. What is the value of this product?</p>"""
    number_string = "7316717653133062491922511967442657474235534919493496983520312774506326239578318016984801869478851843858615607891129494954595017379583319528532088055111254069874715852386305071569329096329522744304355766896648950445244523161731856403098711121722383113622298934233803081353362766142828064444866452387493035890729629049156044077239071381051585930796086670172427121883998797908792274921901699720888093776657273330010533678812202354218097512545405947522435258490771167055601360483958644670632441572215539753697817977846174064955149290862569321978468622482839722413756570560574902614079729686524145351004748216637048440319989000889524345065854122758866688116427171479924442928230863465674813919123162824586178664583591245665294765456828489128831426076900422421902267105562632111110937054421750694165896040807198403850962455444362981230987879927244284909188845801561660979191338754992005240636899125607176060588611646710940507754100225698315520005593572972571636269561882670428252483600823257530420752963450"

In [2]:
p = P008()
p.describe()

## Problem 8: Largest Product in a Series

<p>The four adjacent digits in the $1000$-digit number that have the greatest product are $9 \times 9 \times 8 \times 9 = 5832$.</p><p class="monospace break_word copy_to_clipboard">
731671765313306249192251196744265747423553491949349698352031277450632623957831801698480186947885184385861560789112949495459501737958331952853208805511125406987471585238630507156932909632952274430435576689664895044524452316173185640309871112172238311362229893423380308135336276614282806444486645238749303589072962904915604407723907138105158593079608667017242712188399879790879227492190169972088809377665727333001053367881220235421809751254540594752243525849077116705560136048395864467063244157221553975369781797784617406495514929086256932197846862248283972241375657056057490261407972968652414535100474821663704844031<span class="red">9989</span>000889524345065854122758866688116427171479924442928230863465674813919123162824586178664583591245665294765456828489128831426076900422421902267105562632111110937054421750694165896040807198403850962455444362981230987879927244284909188845801561660979191338754992005240636899125607176060588611646710940507754100225698315520005593572972571636269561882670428252483600823257530420752963450</p><p>Find the thirteen adjacent digits in the $1000$-digit number that have the greatest product. What is the value of this product?</p>

### Solution notes
First we loop through all combinations of 13 digits to see what we're dealing with before optimising. Though some optimisations are immediately apparent, they will be done in the next step. It also turns out numba can't handle converting strings to int, so this first function does not even use numba.

In [3]:
@solution(P008, first=True, make_fast=False, warmup_args=(P008.number_string,))
def string_brute_force(number_string):
    i = 0
    max_product = 0
    while i + 13 < len(number_string) - 1:
        product = 1
        for digit_index in range(13):
            digit = number_string[i + digit_index]
            product *= int(digit)
            if product > max_product:
                max_product = product
        i += 1
    return max_product

In [4]:
p.test_all()

23514624000 found in 1.119344 ms by string_brute_force (first)


The int() function does not exist in numba, so char2int was added to make this function numba compatible.

In [5]:
@solution(P008, make_fast=True, warmup_args=(P008.number_string,))
def numba_string_brute_force(number_string):

    def char2int(c):
        return ord(c) - 48

    i = 0
    max_product = 0
    while i + 13 < len(number_string) - 1:
        product = 1
        for digit_index in range(13):
            digit = char2int(number_string[i + digit_index])
            product *= digit
            if product > max_product:
                max_product = product
        i += 1
    return max_product

In [6]:
p.test_all()

23514624000 found in 0.303865 ms by numba_string_brute_force
23514624000 found in 1.116840 ms by string_brute_force (first)


The idea here is to interrupt any loop which encounters a 0, as the product is now also 0. For some reason this makes it slower.

In [7]:
@solution(P008, make_fast=True, warmup_args=(P008.number_string,))
def optimised_string_brute_force(number_string):

    def char2int(c):
        return ord(c) - 48

    i = 0
    max_product = 0
    while i + 13 < len(number_string) - 1:
        product = 1
        for digit_index in range(13):
            digit = char2int(number_string[i + digit_index])
            if digit == 0:
                break
            product *= digit
            if product > max_product:
                max_product = product
        i += 1
    return max_product

In [8]:
p.test_all()

23514624000 found in 0.227224 ms by numba_string_brute_force
23514624000 found in 0.142696 ms by optimised_string_brute_force
23514624000 found in 0.807624 ms by string_brute_force (first)


Further optimisation: turn the string into an int[]. This is the first actual speed up

In [9]:
@solution(P008, make_fast=True, warmup_args=(P008.number_string,))
def int_list_brute_force(number_string):

    def char2int(c):
        return ord(c) - 48

    number_list = [char2int(c) for c in number_string]

    i = 0
    max_product = 0
    while i + 13 < len(number_list) - 1:
        product = 1
        for digit_index in range(13):
            digit = number_list[i + digit_index]
            if digit == 0:
                break
            product *= digit
            if product > max_product:
                max_product = product
        i += 1
    return max_product

In [10]:
p.test_all()

23514624000 found in 0.031172 ms by int_list_brute_force
23514624000 found in 0.174351 ms by numba_string_brute_force
23514624000 found in 0.105378 ms by optimised_string_brute_force
23514624000 found in 0.702958 ms by string_brute_force (first)


Now we split the string on every $0$. This gives us a list of substrings, this is all we need to test. We remove the ones which are shorter than 13 digits, leaving us with a list of substrings which could contain the answer to check. For these, we use a running total. Multiplying the first 13 digits, and when we move the window of 13 digits over by 1, we divide by the previous digit that got removed from the window and multiply by the new one. Splitting combined with the moving window means this is the fastest method found.

In [11]:
@solution(P008, best= True, make_fast=True, warmup_args=(P008.number_string,))
def zero_split_running_total(number_string):

    def char2int(c):
        return ord(c) - 48

    split_string = number_string.split("0")
    split_string = [substring for substring in split_string if len(substring) >= 13]

    number_lists = [[char2int(c) for c in substring] for substring in split_string]

    max_product = 0
    for number_list in number_lists:
        index_in_list = 0
        product = np.int64(1)
        while index_in_list + 12 < len(number_list):
            # If this is the first set of 13 in a list
            if index_in_list == 0:
                for digit_index in range(13):
                    # Multiply the first 13 digits to be the initial product
                    product *= number_list[digit_index]
            else:
                # If it is not the first set in a list, divide by the digit we just removed from our window
                product //= number_list[index_in_list - 1]
                # And multiply by the next digit
                product *= number_list[index_in_list + 12]
            if product > max_product:
                max_product = product
            index_in_list += 1
    return max_product

In [12]:
p.test_all()

23514624000 found in 0.031694 ms by int_list_brute_force
23514624000 found in 0.183285 ms by numba_string_brute_force
23514624000 found in 0.118245 ms by optimised_string_brute_force
23514624000 found in 0.752837 ms by string_brute_force (first)
23514624000 found in 0.024322 ms by zero_split_running_total (best)
